## Inventory Diff Report Table

- Help us create a table to compute the difference between the simulation initial conditions and the inventory data

In [38]:
import pandas as pd
import numpy as np
import os

In [39]:
OUTPUT_POSTPROCESSING_DIR_PATH = os.getcwd()
SSP_MODELING_DIR_PATH = os.path.dirname(OUTPUT_POSTPROCESSING_DIR_PATH)
SSP_OUTPUT_DIR_PATH = os.path.join(SSP_MODELING_DIR_PATH, "ssp_run_output")
CW_DATA_DIR_PATH = os.path.join(OUTPUT_POSTPROCESSING_DIR_PATH, "data")

In [40]:
ISO3 = "BGR"
REGION_NAME = "bulgaria"
RUN_DIR_PATH = os.path.join(SSP_OUTPUT_DIR_PATH, "sisepuede_results_sisepuede_run_2025-11-27T16;51;05.119133")

### Load emission targets and ssp outputs dfs

In [41]:
# Load emission targets
emission_targets_df = pd.read_csv(os.path.join(CW_DATA_DIR_PATH, "emission_targets_bulgaria_2022.csv"))
emission_targets_df.head()

,Subsector,Gas,Edgar_Sector,Edgar_Subsector,Edgar_Subsector_Synthetic,Vars,id,BGR,Edgar_Class
0,agrc,CH4,Agriculture,AG - Crops,AG - Crops,emission_co2e_ch4_agrc_anaerobicdom_rice:emiss...,AG - Crops - CH4,0.101736,AG - Crops:CH4
1,agrc,CO2,Agriculture,AG - Crops,AG - Crops,emission_co2e_co2_agrc_biomass_bevs_and_spices...,AG - Crops - CO2,1.071093,AG - Crops:CO2
2,agrc,N2O,Agriculture,AG - Crops,AG - Crops,emission_co2e_n2o_agrc_biomass_burning:emissio...,AG - Crops - N2O,3.236068,AG - Crops:N2O
3,lvst,CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lvst_entferm_buffalo:emissio...,AG - Livestock - CH4,1.068538,AG - Livestock:CH4
4,lsmm,CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lsmm_anaerobic_digester:emis...,AG - Livestock - CH4,1.068538,AG - Livestock:CH4


In [42]:
# Load output data
ssp_output_df = pd.read_csv(os.path.join(RUN_DIR_PATH, 
                                         "sisepuede_results_sisepuede_run_2025-11-27T16;51;05.119133_WIDE_INPUTS_OUTPUTS.csv"))
ssp_output_df.head()

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha
0,0,bulgaria,0,106239.676068,1.752955e+06,10623.967607,796797.570511,132799.595085,478078.542306,743677.732477,...,1.039938,2.099992,1.062913,1.25,1.966460,5.4540,55.765409,14.9633,2.555236,92.81
1,0,bulgaria,1,106238.993420,1.752943e+06,10623.899342,796792.450650,132798.741775,478075.470390,743672.953940,...,1.051730,2.180765,1.204477,1.25,2.296954,5.4032,55.765409,15.1841,2.967399,92.81
2,0,bulgaria,2,106689.192161,1.760372e+06,10668.919216,800168.941207,133361.490201,480101.364724,746824.345126,...,1.011737,2.054411,1.196420,1.25,2.479671,5.6089,55.765409,17.7897,2.826425,92.81
3,0,bulgaria,3,106310.524885,1.754124e+06,10631.052489,797328.936639,132888.156107,478397.361984,744173.674197,...,1.011737,2.106612,2.481176,1.25,1.299027,5.5200,55.765409,18.5596,3.076353,92.81
4,0,bulgaria,4,105367.645627,1.738566e+06,10536.764563,790257.342199,131709.557033,474154.405320,737573.519386,...,1.011737,1.733221,2.747621,1.25,2.005811,6.0000,55.765409,21.2497,2.942655,92.81


### Obtain the ssp output values in the emission targets format

In [43]:
def sum_vars_from_ssp_outputs(
    emission_targets_df: pd.DataFrame,
    ssp_outputs_df: pd.DataFrame,
    vars_col: str = "Vars",
    out_col: str = "ssp_total",
    record_missing_col: str | None = "missing_vars",
    ssp_filter: dict | None = None,
) -> pd.DataFrame:
    """
    For each row in emission_targets_df, split the colon-separated strings in `vars_col`,
    find those columns in ssp_outputs_df, sum their values (over rows & columns), and
    write the total to `out_col` in emission_targets_df.

    Parameters
    ----------
    emission_targets_df : DataFrame
        Must contain a string column `vars_col` with colon-separated names.
    ssp_outputs_df : DataFrame
        Wide table whose columns include the names referenced by `emission_targets_df[vars_col]`.
    vars_col : str
        Column in emission_targets_df with colon-separated variable names.
    out_col : str
        New column to create in emission_targets_df with totals from ssp_outputs_df.
    record_missing_col : str | None
        If provided, creates a column listing any missing vars for each row.
    df2_filter : dict | None
        Optional filters to reduce ssp_outputs_df before summing, e.g.
        {"region": "egypt", "time_period": 7}

    Returns
    -------
    DataFrame
        emission_targets_df with new column `out_col` (and `record_missing_col` if requested).
    """
    # Optionally filter ssp_outputs_df by key=value pairs (e.g., region/time_period)
    if ssp_filter:
        mask = pd.Series(True, index=ssp_outputs_df.index)
        for k, v in ssp_filter.items():
            mask &= (ssp_outputs_df[k] == v)
        ssp_view = ssp_outputs_df.loc[mask]
    else:
        ssp_view = ssp_outputs_df

    # Ensure we only operate on numeric data when summing
    numeric_cols = set(ssp_view.select_dtypes(include=[np.number]).columns)

    def _total_for_vars(vars_str: str):
        if pd.isna(vars_str) or not str(vars_str).strip():
            return np.nan, []

        # Split, strip, and deduplicate while preserving order
        raw = [s.strip() for s in str(vars_str).split(":") if s.strip()]
        seen = set()
        cols = [c for c in raw if not (c in seen or seen.add(c))]

        present = [c for c in cols if c in ssp_view.columns and c in numeric_cols]
        missing = [c for c in cols if c not in ssp_view.columns or c not in numeric_cols]

        if not present or ssp_view.empty:
            return np.nan, missing

        # Sum over all filtered rows & all present columns
        vals = ssp_view[present].to_numpy(dtype=float, copy=False)
        total = np.nansum(vals)
        return float(total), missing

    totals, missings = [], []
    for v in emission_targets_df[vars_col].astype("string"):
        total, missing = _total_for_vars(v)
        totals.append(total)
        missings.append(missing)

    emission_targets_df = emission_targets_df.copy()
    emission_targets_df[out_col] = totals
    if record_missing_col is not None:
        emission_targets_df[record_missing_col] = missings

    return emission_targets_df


# -----------------------------
# Example usage
# -----------------------------

# If DF2 has a single row for the target (e.g., region="egypt", a specific time_period):
# df2_filter = {"region": "egypt"}              # or {"region": "egypt", "time_period": 7}
# If you want to sum across all rows of DF2, set df2_filter = None.

# df1_result = sum_vars_from_df2(DF1, DF2, vars_col="Vars",
#                                out_col="DF2_total",
#                                record_missing_col="Missing_in_DF2",
#                                df2_filter={"region": "egypt"})
# print(df1_result.head())


In [44]:
ssp_output_df.head()

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha
0,0,bulgaria,0,106239.676068,1.752955e+06,10623.967607,796797.570511,132799.595085,478078.542306,743677.732477,...,1.039938,2.099992,1.062913,1.25,1.966460,5.4540,55.765409,14.9633,2.555236,92.81
1,0,bulgaria,1,106238.993420,1.752943e+06,10623.899342,796792.450650,132798.741775,478075.470390,743672.953940,...,1.051730,2.180765,1.204477,1.25,2.296954,5.4032,55.765409,15.1841,2.967399,92.81
2,0,bulgaria,2,106689.192161,1.760372e+06,10668.919216,800168.941207,133361.490201,480101.364724,746824.345126,...,1.011737,2.054411,1.196420,1.25,2.479671,5.6089,55.765409,17.7897,2.826425,92.81
3,0,bulgaria,3,106310.524885,1.754124e+06,10631.052489,797328.936639,132888.156107,478397.361984,744173.674197,...,1.011737,2.106612,2.481176,1.25,1.299027,5.5200,55.765409,18.5596,3.076353,92.81
4,0,bulgaria,4,105367.645627,1.738566e+06,10536.764563,790257.342199,131709.557033,474154.405320,737573.519386,...,1.011737,1.733221,2.747621,1.25,2.005811,6.0000,55.765409,21.2497,2.942655,92.81


In [45]:
emission_targets_df_extended = sum_vars_from_ssp_outputs(emission_targets_df, ssp_output_df, vars_col="Vars",
                               out_col="ssp_emission",
                               record_missing_col="missing_in_ssp_outputs",
                               ssp_filter={"region": REGION_NAME, "primary_id": 0, "time_period": 7})

emission_targets_df_extended.head()

,Subsector,Gas,Edgar_Sector,Edgar_Subsector,Edgar_Subsector_Synthetic,Vars,id,BGR,Edgar_Class,ssp_emission,missing_in_ssp_outputs
0,agrc,CH4,Agriculture,AG - Crops,AG - Crops,emission_co2e_ch4_agrc_anaerobicdom_rice:emiss...,AG - Crops - CH4,0.101736,AG - Crops:CH4,0.205505,[]
1,agrc,CO2,Agriculture,AG - Crops,AG - Crops,emission_co2e_co2_agrc_biomass_bevs_and_spices...,AG - Crops - CO2,1.071093,AG - Crops:CO2,1.043769,[]
2,agrc,N2O,Agriculture,AG - Crops,AG - Crops,emission_co2e_n2o_agrc_biomass_burning:emissio...,AG - Crops - N2O,3.236068,AG - Crops:N2O,3.367015,[]
3,lvst,CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lvst_entferm_buffalo:emissio...,AG - Livestock - CH4,1.068538,AG - Livestock:CH4,1.608673,[]
4,lsmm,CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lsmm_anaerobic_digester:emis...,AG - Livestock - CH4,1.068538,AG - Livestock:CH4,0.421244,[]


### Create diff report

In [46]:
# subset the emission targets to create the diff report template
diff_report_df = emission_targets_df_extended[[
    "Subsector",
    "Edgar_Class",
    ISO3,
    "ssp_emission",
]].copy()
diff_report_df.head()

,Subsector,Edgar_Class,BGR,ssp_emission
0,agrc,AG - Crops:CH4,0.101736,0.205505
1,agrc,AG - Crops:CO2,1.071093,1.043769
2,agrc,AG - Crops:N2O,3.236068,3.367015
3,lvst,AG - Livestock:CH4,1.068538,1.608673
4,lsmm,AG - Livestock:CH4,1.068538,0.421244


In [47]:
# merge subsector an id into a single column for clarity
diff_report_df["subsector_id"] = diff_report_df["Subsector"] + " - " + diff_report_df["Edgar_Class"]
diff_report_df = diff_report_df.drop(columns=["Subsector", "Edgar_Class"])
diff_report_df.head()

,BGR,ssp_emission,subsector_id
0,0.101736,0.205505,agrc - AG - Crops:CH4
1,1.071093,1.043769,agrc - AG - Crops:CO2
2,3.236068,3.367015,agrc - AG - Crops:N2O
3,1.068538,1.608673,lvst - AG - Livestock:CH4
4,1.068538,0.421244,lsmm - AG - Livestock:CH4


In [48]:
#rename region column
diff_report_df = diff_report_df.rename(columns={ISO3: "inventory_emission"})
diff_report_df.head()

,inventory_emission,ssp_emission,subsector_id
0,0.101736,0.205505,agrc - AG - Crops:CH4
1,1.071093,1.043769,agrc - AG - Crops:CO2
2,3.236068,3.367015,agrc - AG - Crops:N2O
3,1.068538,1.608673,lvst - AG - Livestock:CH4
4,1.068538,0.421244,lsmm - AG - Livestock:CH4


In [49]:
# Create inventory_share column
diff_report_df["inventory_share"] = diff_report_df["inventory_emission"] / diff_report_df["inventory_emission"].sum()
diff_report_df

,inventory_emission,ssp_emission,subsector_id,inventory_share
0,0.101736,0.205505,agrc - AG - Crops:CH4,0.001740
1,1.071093,1.043769,agrc - AG - Crops:CO2,0.018323
2,3.236068,3.367015,agrc - AG - Crops:N2O,0.055357
3,1.068538,1.608673,lvst - AG - Livestock:CH4,0.018279
4,1.068538,0.421244,lsmm - AG - Livestock:CH4,0.018279
5,0.120672,0.217645,lsmm - AG - Livestock:N2O,0.002064
6,0.000000,0.000000,ccsq - CCSQ:CH4,0.000000
7,0.000000,0.000000,ccsq - CCSQ:CO2,0.000000
8,0.000000,0.000000,ccsq - CCSQ:N2O,0.000000
9,0.415001,0.015267,scoe - EN - Building:CH4,0.007099


In [50]:
# Calculate error column, avoid division by zero by adding a small constant to the denominator
epsilon = 1e-8
diff_report_df["rel_error"] = (diff_report_df["ssp_emission"] - diff_report_df["inventory_emission"]).abs() / (diff_report_df["inventory_emission"] + epsilon)
diff_report_df["abs_error"] = (diff_report_df["ssp_emission"] - diff_report_df["inventory_emission"]).abs()
diff_report_df.head()

,inventory_emission,ssp_emission,subsector_id,inventory_share,rel_error,abs_error
0,0.101736,0.205505,agrc - AG - Crops:CH4,0.001740,1.019981,0.103769
1,1.071093,1.043769,agrc - AG - Crops:CO2,0.018323,0.025510,0.027324
2,3.236068,3.367015,agrc - AG - Crops:N2O,0.055357,0.040465,0.130947
3,1.068538,1.608673,lvst - AG - Livestock:CH4,0.018279,0.505490,0.540135
4,1.068538,0.421244,lsmm - AG - Livestock:CH4,0.018279,0.605775,0.647294


In [51]:
# Set subsector_id at the beginning of the df
diff_report_df = diff_report_df[[
    "subsector_id",
    "inventory_emission",
    "ssp_emission",
    "inventory_share",
    "abs_error",
    "rel_error"
]]

# sort by squared_error descending
diff_report_df = diff_report_df.sort_values(by="rel_error", ascending=False)
diff_report_df.head(10)

,subsector_id,inventory_emission,ssp_emission,inventory_share,abs_error,rel_error
41,soil - LULUCF - Organic Soil:N2O,0.000000,16.266280,0.000000,16.266280,1.626628e+09
37,frst - LULUCF - Forest Land Removals:CO2,0.000000,9.917352,0.000000,9.917352,9.917352e+08
39,frst - LULUCF - HWP:CO2,0.000000,-2.616273,0.000000,2.616273,2.616273e+08
36,frst - LULUCF - Forest Land:CH4,0.000000,0.110324,0.000000,0.110324,1.103239e+07
34,lndu - LULUCF - Deforestation:CH4,0.000000,0.091378,0.000000,0.091378,9.137757e+06
35,lndu - LULUCF - Deforestation:CO2,0.111938,5.812733,0.001915,5.700795,5.092831e+01
45,waso - Waste - Solid Waste:CO2,0.006775,0.184991,0.000116,0.178215,2.630378e+01
40,soil - LULUCF - Organic Soil:CO2,0.080627,1.634720,0.001379,1.554093,1.927502e+01
48,trww - Waste - Wastewater Treatment:N2O,0.097626,0.760928,0.001670,0.663301,6.794297e+00
30,ippu - IN - Industrial Processes:N2O,0.260753,0.938119,0.004461,0.677365,2.597724e+00


In [52]:
diff_report_df.tail(40)

,subsector_id,inventory_emission,ssp_emission,inventory_share,abs_error,rel_error
30,ippu - IN - Industrial Processes:N2O,0.260753,0.938119,0.004461,0.677365,2.597724
20,inen - EN - Manufacturing/Construction:N2O,0.023925,0.053136,0.000409,0.029211,1.220941
15,fgtv - EN - Fugitive Emissions:CH4,1.125722,2.388122,0.019257,1.262401,1.121415
0,agrc - AG - Crops:CH4,0.101736,0.205505,0.001740,0.103769,1.019981
9,scoe - EN - Building:CH4,0.415001,0.015267,0.007099,0.399734,0.963212
33,ippu - IN - Industrial Processes:SF6,0.019636,0.002130,0.000336,0.017506,0.891513
24,trns - EN - Transportation:CH4,0.050660,0.006344,0.000867,0.044316,0.874778
5,lsmm - AG - Livestock:N2O,0.120672,0.217645,0.002064,0.096973,0.803609
25,trns - EN - Transportation:CO2,10.314811,2.500371,0.176449,7.814441,0.757594
12,entc - EN - Electricity/Heat:CH4,0.023684,0.006468,0.000405,0.017216,0.726919


### Save diff table

In [53]:
diff_report_df.to_csv(os.path.join(RUN_DIR_PATH, f"inventory_diff_report_{REGION_NAME}.csv"), index=False)